# GPT Text Generation: Local GPT-2 and the OpenAI API

## 📚 Learning Objectives

By completing this notebook, you will:
- **Generate real text with a GPT model (GPT-2) running locally** via Hugging Face — no API key needed
- Compare **greedy decoding vs. sampling**, and see how **temperature** changes the generated text
- Understand basic **prompt engineering** ideas
- See how commercial APIs (OpenAI) scale the same idea up — as a **code walkthrough, not executed here**
  (running it requires a paid OpenAI account and API key)

## 🔗 Prerequisites

- ✅ `01_attention_transformers_bridge.ipynb` — GPT as a *causal* (left-to-right) transformer
- ✅ `03_bert_advanced_usage.ipynb` — Hugging Face `pipeline` usage

---

## Introduction

**GPT (Generative Pre-trained Transformer)** models generate text one token at a time, each token predicted
from everything before it (the causal mask you visualized in the bridge notebook). In this notebook we
actually run **GPT-2** — OpenAI's openly released 2019 model — on your own machine. It is small and dated
compared with GPT-4-class models, but it is the same architecture family, and watching it write makes
temperature and sampling concrete. Afterwards we walk through how the modern OpenAI API exposes far larger
models through the same concepts.


## 📥 Inputs & 📤 Outputs

**Inputs:** short text prompts, and the pretrained `gpt2` model from Hugging Face (~500MB download on first
run; cached afterwards).

**Outputs:** real generated text (greedy and sampled at different temperatures), plus a *non-executed* code
walkthrough of the OpenAI API.

---


In [1]:
# Setup: Hugging Face pipeline for LOCAL GPT-2 generation (no API key, no cost).
# The OpenAI client is optional — Part 3 only walks through its code without calling it.

# Local text generation uses Hugging Face Transformers (PyTorch backend)
try:
    from transformers import pipeline, set_seed
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()          # keep the output clean
    HAS_TRANSFORMERS = True
    print("✅ Hugging Face Transformers available — local GPT-2 generation will run")
except ImportError:
    HAS_TRANSFORMERS = False
    print("⚠️  Transformers not available. Install with: pip install transformers torch")

# The OpenAI client is OPTIONAL: Part 3 is a code walkthrough only and does not call the API
try:
    import openai  # noqa: F401
    HAS_OPENAI = True
    print("✅ OpenAI client library installed (not required for this notebook)")
except ImportError:
    HAS_OPENAI = False
    print("ℹ️  OpenAI client library not installed — that's fine; Part 3 is a walkthrough only")

✅ Hugging Face Transformers available — local GPT-2 generation will run
ℹ️  OpenAI client library not installed — that's fine; Part 3 is a walkthrough only


## Part 1: Understanding GPT Models

In [2]:
# Concept cell: print a map of the GPT model family before we run one.
# Knowing the scale ladder (GPT-2's 124M -> GPT-3's 175B parameters) explains why
# the local model below is fluent but far from ChatGPT-quality.

print("=" * 60)
print("GPT Models Overview")
print("=" * 60)

print("\nGPT Model Characteristics:")
print("  - Generative: Can generate new text")
print("  - Pre-trained: Trained on large text corpora")
print("  - Transformer-based: Uses attention mechanisms")
print("  - Autoregressive: Generates text token by token")

print("\n" + "-" * 60)
print("GPT-3 vs GPT-4:")
print("-" * 60)
print("  GPT-3:")
print("    - 175 billion parameters")
print("    - Text generation, completion, Q&A")
print("    - Available via OpenAI API")
print("  GPT-4:")
print("    - Larger and more capable")
print("    - Better reasoning and instruction following")
print("    - Multimodal capabilities (text + images)")

print("\n✅ GPT models are powerful for:")
print("  - Text generation")
print("  - Text completion")
print("  - Question answering")
print("  - Code generation")
print("  - Creative writing")

GPT Models Overview

GPT Model Characteristics:
  - Generative: Can generate new text
  - Pre-trained: Trained on large text corpora
  - Transformer-based: Uses attention mechanisms
  - Autoregressive: Generates text token by token

------------------------------------------------------------
GPT-3 vs GPT-4:
------------------------------------------------------------
  GPT-3:
    - 175 billion parameters
    - Text generation, completion, Q&A
    - Available via OpenAI API
  GPT-4:
    - Larger and more capable
    - Better reasoning and instruction following
    - Multimodal capabilities (text + images)

✅ GPT models are powerful for:
  - Text generation
  - Text completion
  - Question answering
  - Code generation
  - Creative writing


## Part 2: Generating Text with a Local GPT-2 (no API key)

Two decoding strategies, both run for real below:

- **Greedy decoding** (`do_sample=False`): always pick the single most likely next token. Deterministic —
  and famously prone to getting stuck repeating itself.
- **Sampling with temperature** (`do_sample=True`): draw the next token from the probability distribution.
  **Temperature** rescales that distribution before drawing: lower values (< 1) concentrate probability on
  the top tokens (safer, more predictable text), higher values (> 1) flatten it (more surprising, more
  chaotic text).

We fix the random seed so the sampled outputs below are reproducible.


In [3]:
# Generate text with GPT-2 three ways: greedy, cool sampling, hot sampling.
# The decoding strategy changes the output as much as the model does —
# same weights, very different text.

if HAS_TRANSFORMERS:
    print("Loading GPT-2 (124M parameters)... first run downloads ~500MB")
    generator = pipeline("text-generation", model="gpt2")
    print("✅ GPT-2 loaded\n")

    prompt = "Natural language processing is"
    print("=" * 70)
    print(f"Prompt: {prompt!r}")
    print("=" * 70)

    # --- Greedy decoding: deterministic, often repetitive -------------------
    greedy = generator(prompt, max_new_tokens=35, do_sample=False,
                       num_return_sequences=1, pad_token_id=50256)
    print("\n--- Greedy decoding (always the most likely next token) ---")
    print(greedy[0]["generated_text"])

    # --- Sampling at two temperatures --------------------------------------
    for temp in (0.7, 1.3):
        set_seed(42)
        outs = generator(prompt, max_new_tokens=35, do_sample=True, temperature=temp,
                         num_return_sequences=2, pad_token_id=50256)
        print(f"\n--- Sampling, temperature={temp} (2 samples) ---")
        for k, o in enumerate(outs, 1):
            print(f"[{k}] {o['generated_text']}\n")

    print("=" * 70)
    print("All of the text above was genuinely generated by GPT-2 on this machine.")
    print("Compare the temperature=0.7 samples with the temperature=1.3 ones:")
    print("higher temperature spreads probability over more tokens, so the text")
    print("takes more risks. GPT-2 (2019, 124M params) also happily writes fluent")
    print("nonsense — factual reliability is NOT what small LMs give you.")
else:
    print("Install transformers + torch to run local GPT-2 generation:")
    print("  pip install transformers torch")

Loading GPT-2 (124M parameters)... first run downloads ~500MB


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ GPT-2 loaded

Prompt: 'Natural language processing is'



--- Greedy decoding (always the most likely next token) ---
Natural language processing is a very important part of the language learning process.

The first step is to understand the language. The second step is to understand the language.

The first step



--- Sampling, temperature=0.7 (2 samples) ---
[1] Natural language processing is now more secure and easier to implement.

The new language provides better performance for the GPU, allowing for the best possible Carson's-style performance.

The new

[2] Natural language processing is the most powerful tool available to a computer programmer. Language processing is an advanced form of computer vision and translation. It helps computer vision experts understand and match words and phrases without using




--- Sampling, temperature=1.3 (2 samples) ---
[1] Natural language processing is now more secure and easier to implement and use."

While it may seem an insignificant thing until you consider the fact that Carson's efforts and contributions were not limited to just

[2] Natural language processing is the best way to quickly create, share and collaborate with code. Not everybody will enjoy it as its a tool to generate code, but in my own practice, I found that

All of the text above was genuinely generated by GPT-2 on this machine.
Compare the temperature=0.7 samples with the temperature=1.3 ones:
higher temperature spreads probability over more tokens, so the text
takes more risks. GPT-2 (2019, 124M params) also happily writes fluent
nonsense — factual reliability is NOT what small LMs give you.


## Part 3: Scaling Up — the OpenAI API (walkthrough, not executed)

GPT-4-class models are far too large to run locally; you reach them through an API. The concepts are the
ones you just used — a prompt, `max_tokens`, `temperature` — plus a chat message format.

⚠️ **This section is a code walkthrough only.** The cell below *prints* the code instead of running it,
because executing it needs an OpenAI account, an `OPENAI_API_KEY` environment variable, and paid credits.
If you have a key, paste the printed code into a new cell and run it.


In [4]:
# Walkthrough (NOT executed): the modern OpenAI chat-completions API pattern.
# The parameters mirror what you just used locally — model, max tokens,
# temperature — plus the system/user message format used by all chat LLMs.

print("=" * 70)
print("Modern OpenAI API usage (openai>=1.0) — WALKTHROUGH, not executed")
print("=" * 70)
print('''
    # 1. Install the client:      pip install openai
    # 2. Set your key (never hardcode it):
    #      export OPENAI_API_KEY="sk-..."

    from openai import OpenAI

    client = OpenAI()                      # reads OPENAI_API_KEY from the environment

    response = client.chat.completions.create(
        model="gpt-4o-mini",               # or another available chat model
        messages=[
            {"role": "system", "content": "You are a helpful NLP tutor."},
            {"role": "user",   "content": "Explain tokenization in two sentences."},
        ],
        max_tokens=100,
        temperature=0.7,
    )

    print(response.choices[0].message.content)
''')

print("Key parameters (same ideas you used with GPT-2 above):")
print("  - model:       which model answers (capability vs. cost trade-off)")
print("  - messages:    the conversation so far (system / user / assistant roles)")
print("  - max_tokens:  cap on the generated length")
print("  - temperature: 0 = focused/deterministic ... ~1 = balanced ... higher = adventurous")

print("\nPrompt engineering tips (apply to any LLM):")
print("  - Be specific about the task, audience, and output format")
print("  - Provide examples (few-shot prompting)")
print("  - Give the model a role via the system message")
print("  - Iterate: small prompt changes can change results a lot")

Modern OpenAI API usage (openai>=1.0) — WALKTHROUGH, not executed

    # 1. Install the client:      pip install openai
    # 2. Set your key (never hardcode it):
    #      export OPENAI_API_KEY="sk-..."

    from openai import OpenAI

    client = OpenAI()                      # reads OPENAI_API_KEY from the environment

    response = client.chat.completions.create(
        model="gpt-4o-mini",               # or another available chat model
        messages=[
            {"role": "system", "content": "You are a helpful NLP tutor."},
            {"role": "user",   "content": "Explain tokenization in two sentences."},
        ],
        max_tokens=100,
        temperature=0.7,
    )

    print(response.choices[0].message.content)

Key parameters (same ideas you used with GPT-2 above):
  - model:       which model answers (capability vs. cost trade-off)
  - messages:    the conversation so far (system / user / assistant roles)
  - max_tokens:  cap on the generated length
  - temperatu

## Summary

### What you actually did here
- Ran **real text generation with GPT-2 locally** — greedy decoding and sampling at temperatures 0.7 and 1.3
- Saw generation parameters (`max_new_tokens`, `temperature`, seeding) affect actual outputs
- Walked through (without executing) the modern OpenAI API call for GPT-4-class models

### Key Concepts
1. **Autoregressive generation**: GPT predicts one token at a time, left to right (the causal mask from the bridge notebook)
2. **Decoding strategy matters**: greedy is deterministic but repetitive; sampling + temperature trades predictability for creativity
3. **Local vs. API**: small open models (GPT-2) run on your machine; frontier models are reached through paid APIs, with the same core parameters
4. **Prompt engineering**: clear, specific, example-rich prompts get better completions

### Honest limitations
- GPT-2 is a 2019-era 124M-parameter model: fluent, but unreliable on facts — treat its output as a mechanism demo, not information
- The OpenAI section was **not executed** in this notebook; nothing here required an API key

**Reference:** Course 07, Unit 4: "Deep Learning for NLP" — GPT text generation practical content


## 📚 References

The GPT lineage, plus the decoding-strategy paper behind the temperature experiments above:

1. Radford, A., Wu, J., Child, R., et al. (2019). *Language Models are Unsupervised Multitask Learners* (GPT-2). OpenAI Technical Report.
2. Brown, T. B., Mann, B., Ryder, N., et al. (2020). *Language Models are Few-Shot Learners* (GPT-3). NeurIPS 2020. [arXiv:2005.14165](https://arxiv.org/abs/2005.14165)
3. Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y. (2020). *The Curious Case of Neural Text Degeneration*. ICLR 2020. [arXiv:1904.09751](https://arxiv.org/abs/1904.09751)
4. Ouyang, L., Wu, J., Jiang, X., et al. (2022). *Training Language Models to Follow Instructions with Human Feedback* (InstructGPT). NeurIPS 2022. [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
